In [0]:
%run "/Workspace/Users/collinlee-@live.ca/fake_general_ledger/2.1 silver_transformation_framework"

In [0]:
# ---------------------------------------------------------
# 0. Silver Transformation Framework Parameters
# ---------------------------------------------------------

schema = StructType([
    StructField("territory_key", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("region", StringType(), True)
])

bronze_table = "fake_general_ledger.default.bronze_territory"

silver_table = "fake_general_ledger.default.silver_territory"  

merge_condition = """target.territory_key = source.territory_key"""

required_columns = [
    "territory_key",
    "country",
    "region"
]

primary_key_columns = ["territory_key"]


order_column = "territory_key"



In [0]:
# ---------------------------------------------------------
# 1. Silver Transformation Driver Code
# ---------------------------------------------------------

bronze_df = spark.table(
    bronze_table
)

bronze_df = standardize_column_names(
    bronze_df
)

silver_df = apply_silver_schema(
    bronze_df,
    schema
)

silver_df, invalid_df = validate_required_columns(
    silver_df,
    required_columns
)

silver_df = deduplicate_records(
    silver_df,
    primary_key_columns,
    order_column
)

silver_df = apply_silver_metadata(
    silver_df
)

silver_df = rename_bronze_metadata(
    silver_df
)

silver_df = drop_excess_bronze_metadata(
    silver_df
)

merge_to_silver(
    spark,
    silver_df,
    silver_table,
    merge_condition
)